# Figure 7 | Multi-scale GNN vulnerability modeling

Cleaned notebook for the public NVU AD spatial transcriptomics repository. Paths are repository-relative; set `NVU_PROJECT_ROOT` if running from another working directory.


In [ ]:
# Repository-relative paths
import os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("NVU_PROJECT_ROOT", Path.cwd().parent)).resolve()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
REFERENCE_DIR = PROJECT_ROOT / "references"
FIGURE_DIR = RESULTS_DIR / "figure7"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os, gc, sys, time, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy.spatial import KDTree
from scipy.sparse import issparse
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from torch_geometric.data import Data, Batch as PyGBatch
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch.optim.lr_scheduler import CosineAnnealingLR
warnings.filterwarnings('ignore')
from joblib import Parallel, delayed
rng = np.random.default_rng(42)
import torch
# 必须在任何PyTorch操作之前，且只能调用一次
torch.set_num_threads(88)
torch.set_num_interop_threads(40)
print(f"线程设置完成: {torch.get_num_threads()} threads")
GNN_DIR = '../data/figure7'
CELLTYPE_ORDER = ['Neuron','Astro','Micro','Endo','Pericyte','Oligo','OPC']
REGION_MAP = {
    'CA1':0,'CA2':1,'CA3':2,'CA4':3,'DG':4,'FAS':5,'SLRM':6,
    'L1':7,'L23':8,'L456':9,'WM':10
}

# 之后再加载.py文件
with open('../scripts/figure7_model.py',
          'r', encoding='utf-8') as f:
    code = f.read()
code_defs = code.split("if __name__ == '__main__':")[0]
exec(code_defs, globals())
print("函数定义载入完成 ✓")

### 数据预处理

In [ ]:
from pathlib import Path
import pickle
import pandas as pd

# ======================================================
# 0. 路径参数：按你服务器实际目录修改这里
# ======================================================
HIP_DIR = '../data/figure7/Hip'
CTX_DIR = '../data/figure7/Cortex'   # 如果你的皮层不是这个路径，请改这里
GNN_DIR = '../data/figure7'
PLOT_DIR = '../results/figure7'

FORCE_REGENERATE = True
N_JOBS = 12   # 先别用36，避免并行读h5ad时更容易出错

print('HIP_DIR =', HIP_DIR)
print('CTX_DIR =', CTX_DIR)

hip_files = sorted(Path(HIP_DIR).glob('*.h5ad'))
ctx_files = sorted(Path(CTX_DIR).glob('*.h5ad'))

print(f'HIP h5ad 文件数: {len(hip_files)}')
print(f'CTX h5ad 文件数: {len(ctx_files)}')

if len(hip_files) == 0:
    raise FileNotFoundError(f'HIP_DIR 下没有 h5ad: {HIP_DIR}')
if len(ctx_files) == 0:
    raise FileNotFoundError(f'CTX_DIR 下没有 h5ad: {CTX_DIR}')

# ======================================================
# 1. 读取基因列表
# ======================================================
modulegens_hip = pd.read_csv('../data/figure7/NVU.Module.csv')
modulegens_cortex = pd.read_csv('../data/figure7/Cortex_up_NVU.Module.csv')
Abetagenes = pd.read_csv('../data/figure7/Abeta_associated_genes_intersections.csv')

gene_map_hip = build_gene_module_map(
    modulegens_hip, modulegens_cortex, Abetagenes, 'hip'
)
gene_map_ctx = build_gene_module_map(
    modulegens_hip, modulegens_cortex, Abetagenes, 'ctx'
)

# ======================================================
# 2. 读取 LR 数据
# ======================================================
lr_hip = pd.read_csv('../data/figure7/Hip_all_Stereosite.csv')
lr_ctx = pd.read_csv('../data/figure7/Cortex_all_Stereosite.csv')
lr_ctx = lr_ctx[~lr_ctx['sample'].isin(CTX_EXCLUDE)].copy()

print('海马 LR:')
TOP_LR_HIP = select_top_lr_pairs(lr_hip)
print('皮层 LR:')
TOP_LR_CTX = select_top_lr_pairs(lr_ctx)

lr_feat_hip = build_lr_feature_matrix(lr_hip, TOP_LR_HIP)
lr_feat_ctx = build_lr_feature_matrix(lr_ctx, TOP_LR_CTX)

LR_NAMES_HIP = [
    f'LR__{p}__{s}'
    for p in TOP_LR_HIP
    for s in ['mean', 'max']
]
LR_NAMES_CTX = [
    f'LR__{p}__{s}'
    for p in TOP_LR_CTX
    for s in ['mean', 'max']
]

SCALAR_NAMES_HIP = get_full_scalar_names(
    gene_map_hip, CELLTYPE_ORDER, LR_NAMES_HIP
)
SCALAR_NAMES_CTX = get_full_scalar_names(
    gene_map_ctx, CELLTYPE_ORDER, LR_NAMES_CTX
)

# ======================================================
# 3. 强制重新生成 all_results_v2.pkl
# ======================================================
pkl_path = f'{GNN_DIR}/all_results_v2.pkl'

if FORCE_REGENERATE and Path(pkl_path).exists():
    print(f'删除旧缓存: {pkl_path}')
    Path(pkl_path).unlink()

print('开始重新生成 all_results_v2.pkl ...')
all_results = generate_all_results(
    gene_map_hip, gene_map_ctx,
    SCALAR_NAMES_HIP, SCALAR_NAMES_CTX,
    lr_feat_hip, lr_feat_ctx,
    LR_NAMES_HIP, LR_NAMES_CTX,
    n_jobs=N_JOBS
)

# ======================================================
# 4. 检查新缓存
# ======================================================
def cache_is_new(results):
    return all(
        'node_gene_names' in r and 'node_lr_names' in r
        for r in results
    )

has_gene_cache = cache_is_new(all_results)
print(f'缓存包含统一基因/LR列信息: {has_gene_cache}')

if not has_gene_cache:
    raise RuntimeError(
        '重新生成后仍缺少 node_gene_names/node_lr_names。'
        '请确认 ../scripts/figure7_model.py 是新版。'
    )

n_hip = sum(r['tissue'] == 'hip' for r in all_results)
n_ctx = sum(r['tissue'] == 'ctx' for r in all_results)

print(f'总样本={len(all_results)}, HIP={n_hip}, CTX={n_ctx}')
print(f'总NVU={sum(r["n_nvu"] for r in all_results)}')

print('\nHIP 样本:')
for s in sorted(r['sample_id'] for r in all_results if r['tissue'] == 'hip'):
    print(s)

print('\nCTX 样本:')
for s in sorted(r['sample_id'] for r in all_results if r['tissue'] == 'ctx'):
    print(s)


### 训练

In [ ]:
# # =======================================================
# # Figure7 main analysis workflow
# # 目标：提高AUC + 输出敏感基因、敏感LR、易感脑区、NVU级分类
# # =======================================================
# # ── QC过滤参数：你可以在这里手动排除质量差/标签可疑/文件损坏样本 ──

# EXCLUDE_SAMPLE_IDS = ['GSM8330063_A02092E1','GSM8330071_B01806B5', #'GSM8330061_B02008D2','GSM8330067_B01809C2','GSM8330060_B02009F6',
#                       'D01574C2','D01574C6', 'B03421A5','B03421A6','control_1','control_2']# ,'B03421F4','D03556E6'
# # EXCLUDE_SAMPLE_IDS =[]
# MIN_NVU = None   # 例如 20；None 表示不按NVU数量过滤
# MAX_NVU = None   # 例如 5000；None 表示不按NVU数量过滤

# analysis_results, qc_removed_df = filter_all_results(
#     all_results,
#     exclude_sample_ids=EXCLUDE_SAMPLE_IDS,
#     min_nvu=MIN_NVU,
#     max_nvu=MAX_NVU,
# )
# if len(qc_removed_df):
#     qc_removed_df.to_csv(f'{PLOT_DIR}/qc_removed_samples.csv', index=False)
# print('=' * 55)
# print('1) 计算 AD/Control 敏感基因和敏感 LR，并生成训练权重')
# hip_weights, hip_region_pre = export_sensitivity_tables(
#     analysis_results,
#     tissue='hip',
#     out_dir=PLOT_DIR,
#     weight_strength=2.5,
#     max_weight=6.0,
# )
# ctx_weights, ctx_region_pre = export_sensitivity_tables(
#     analysis_results,
#     tissue='ctx',
#     out_dir=PLOT_DIR,
#     weight_strength=2.5,
#     max_weight=6.0,
# )

# print('\nTop sensitive HIP genes:')
# display(hip_weights['gene_table'].head(20))
# print('\nTop sensitive HIP LR pairs:')
# display(hip_weights['lr_table'].head(20))

# print('\n' + '=' * 55)
# print('2) 海马 weighted GNN：gene + sensitive gene + sensitive LR + NVU structure')
# auc_hip_w_all, auc_hip_w_fold, folds_hip_w, hip_w_samples = train_gnn(
#     analysis_results,
#     tissue='hip',
#     n_folds=5,
#     n_epochs=80,
#     lambda_nvu=0.2,
#     max_nvu=500,
#     max_edges_per_nvu=10000,
#     node_feature_mode='all',
#     use_density_scalar=False,
#     feature_weights='auto',
#     gene_input_scale=1.0,
#     lr_input_scale=0.7,
#     structure_input_scale=0.35,
#     gene_l1=5e-4,
#     weight_decay=1e-1,
#     min_loss_threshold=0.10,
# )

# print('\n' + '=' * 55)
# print('3) 海马 weighted gene-only 对照：只看基因信号')
# auc_hip_gene_w_all, auc_hip_gene_w_fold, folds_hip_gene_w, hip_gene_w_samples = train_gnn(
#     analysis_results,
#     tissue='hip',
#     n_folds=5,
#     n_epochs=80,
#     lambda_nvu=0.1,
#     max_nvu=500,
#     max_edges_per_nvu=10000,
#     node_feature_mode='gene_only',
#     use_density_scalar=False,
#     feature_weights='auto',
#     gene_input_scale=1.0,
#     lr_input_scale=0.7,
#     structure_input_scale=0.35,
#     gene_l1=5e-4,
#     weight_decay=1e-1,
#     min_loss_threshold=0.10,
# )

# print('\n' + '=' * 55)
# print('4) 皮层 weighted GNN：LOSO，敏感 gene + LR 加权')
# auc_ctx_w_all, auc_ctx_w_fold, folds_ctx_w, ctx_w_samples = train_gnn(
#     analysis_results,
#     tissue='ctx',
#     n_folds=5,
#     n_epochs=100,
#     lambda_nvu=0.2,
#     max_nvu=500,
#     max_edges_per_nvu=10000,
#     node_feature_mode='all',
#     use_density_scalar=False,
#     feature_weights='auto',
#     gene_input_scale=1.0,
#     lr_input_scale=0.7,
#     structure_input_scale=0.35,
#     gene_l1=5e-4,
#     weight_decay=1e-1,
#     min_loss_threshold=0.10,
# )

# print('\n' + '=' * 55)
# print('5) 根据NVU级预测更新易感脑区表')
# _, hip_region_post = export_sensitivity_tables(
#     analysis_results,
#     tissue='hip',
#     out_dir=PLOT_DIR,
#     weight_strength=2.5,
#     max_weight=6.0,
#     nvu_pred_path=f'{PLOT_DIR}/nvu_pred_hip.csv',
# )
# _, ctx_region_post = export_sensitivity_tables(
#     analysis_results,
#     tissue='ctx',
#     out_dir=PLOT_DIR,
#     weight_strength=2.5,
#     max_weight=6.0,
#     nvu_pred_path=f'{PLOT_DIR}/nvu_pred_ctx.csv',
# )

# print('\nTop susceptible HIP regions:')
# display(hip_region_post.head(20))
# print('\nTop susceptible CTX regions:')
# display(ctx_region_post.head(20))

# print('\n' + '=' * 55)
# print('6) 全量模型训练，用于导出模型内 gene gate 权重')
# model_hip, hip_samples_scaled = train_full_model(
#     hip_w_samples, tissue='hip', n_epochs=100,
#     lambda_nvu=0.2, use_density_scalar=False, gene_l1=5e-4,
# )
# model_ctx, ctx_samples_scaled = train_full_model(
#     ctx_w_samples, tissue='ctx', n_epochs=100,
#     lambda_nvu=0.2, use_density_scalar=False, gene_l1=5e-4,
# )

# print('\n' + '=' * 55)
# print('结果汇总')
# print(f'海马 weighted gene+LR AUC  = {auc_hip_w_all:.4f} / Fold={auc_hip_w_fold:.4f} ± {np.std(folds_hip_w):.4f}')
# print(f'海马 weighted gene-only AUC = {auc_hip_gene_w_all:.4f} / Fold={auc_hip_gene_w_fold:.4f} ± {np.std(folds_hip_gene_w):.4f}')
# print(f'皮层 weighted gene+LR AUC  = {auc_ctx_w_all:.4f} / Fold={auc_ctx_w_fold:.4f} ± {np.std(folds_ctx_w):.4f}')
# print('输出表格:')
# print(f'  {PLOT_DIR}/sensitive_genes_hip.csv')
# print(f'  {PLOT_DIR}/sensitive_lr_pairs_hip.csv')
# print(f'  {PLOT_DIR}/susceptible_regions_hip.csv')
# print(f'  {PLOT_DIR}/nvu_pred_hip.csv')
# print(f'  {PLOT_DIR}/sensitive_genes_ctx.csv')
# print(f'  {PLOT_DIR}/sensitive_lr_pairs_ctx.csv')
# print(f'  {PLOT_DIR}/susceptible_regions_ctx.csv')
# print(f'  {PLOT_DIR}/nvu_pred_ctx.csv')
# print(f'  {PLOT_DIR}/gene_weight_hip.csv')
# print(f'  {PLOT_DIR}/gene_weight_ctx.csv')



### plotting

In [ ]:
# 之后再加载.py文件
with open('../scripts/figure7_model.py',
          'r', encoding='utf-8') as f:
    code = f.read()
code_defs = code.split("if __name__ == '__main__':")[0]
exec(code_defs, globals())
print("函数定义载入完成 ✓")
# ======================================================
# 因素重要性图：gene modules / LR / NVU structure
# ======================================================
set_publication_plot_style()
factor_df = plot_model_factor_importance(
    plot_dir=PLOT_DIR,
    out_prefix=f'{PLOT_DIR}/figure6_factor_importance_compact',
    all_results=all_results,
    gene_map_hip=gene_map_hip,
    gene_map_ctx=gene_map_ctx,
    top_n_genes=50,
    top_n_lr=50,
    importance_transform='log1p',
    post_normalize_power=0.5,
    fixed_point_size=72,
)

display(factor_df)



In [ ]:
# ======================================================
# 3. 分类任务展示：模型风险空间 + 分组织预测概率分布
#    只读取已有CSV，不重新训练模型
# ======================================================
print('\n绘制模型风险空间图...')
risk_df = plot_risk_embedding_from_csv(
    pred_hip_path=f'{PLOT_DIR}/gnn_pred_hip.csv',
    pred_ctx_path=f'{PLOT_DIR}/gnn_pred_ctx.csv',
    nvu_hip_path=f'{PLOT_DIR}/nvu_pred_hip.csv',
    nvu_ctx_path=f'{PLOT_DIR}/nvu_pred_ctx.csv',
    out_path=f'{PLOT_DIR}/classification_risk_embedding.pdf',
)

print('\n绘制分组织预测概率分布图...')
strip_df = plot_tissue_split_prediction_strip(
    pred_hip_path=f'{PLOT_DIR}/gnn_pred_hip.csv',
    pred_ctx_path=f'{PLOT_DIR}/gnn_pred_ctx.csv',
    out_path=f'{PLOT_DIR}/prediction_score_strip_hip_ctx.pdf',
)

print('\n绘制统一AUC图...')
auc_df = plot_auc_summary(
    {
        'HIP weighted': f'{PLOT_DIR}/gnn_pred_hip.csv',
        'CTX weighted': f'{PLOT_DIR}/gnn_pred_ctx.csv',
    },
    out_path=f'{PLOT_DIR}/auc_summary_hip_ctx.pdf',
)

display(auc_df)


In [ ]:
# ======================================================
# 只画 HIP+CTX 合并 ROC + AUC 柱状图
# 颜色统一绿色，误差棒 = bootstrap 95% CI
# ======================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

PLOT_DIR = '../results/figure7'
GREEN = '#00A087'

pred_hip = pd.read_csv(f'{PLOT_DIR}/gnn_pred_hip.csv')
pred_ctx = pd.read_csv(f'{PLOT_DIR}/gnn_pred_ctx.csv')

pred_hip['tissue'] = 'HIP'
pred_ctx['tissue'] = 'CTX'

pred = pd.concat([pred_hip, pred_ctx], ignore_index=True)

y = pred['label'].astype(int).values
p = pred['pred_prob'].astype(float).values

auc = roc_auc_score(y, p)
fpr, tpr, _ = roc_curve(y, p)

# bootstrap 95% CI
rng = np.random.default_rng(42)
boot_aucs = []
n_boot = 2000
n = len(y)

for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    if len(np.unique(y[idx])) < 2:
        continue
    boot_aucs.append(roc_auc_score(y[idx], p[idx]))

boot_aucs = np.array(boot_aucs)
ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5))

# ROC
axes[0].plot(
    fpr, tpr,
    color=GREEN,
    lw=2.5,
    label=f'HIP+CTX AUC={auc:.3f}'
)
axes[0].plot([0, 1], [0, 1], '--', color='0.7', lw=1)
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].legend(frameon=False, fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# AUC bar
yerr = np.array([[auc - ci_low], [ci_high - auc]])

axes[1].bar(
    [0],
    [auc],
    yerr=yerr,
    capsize=5,
    color=GREEN,
    edgecolor='black',
    linewidth=0.7,
    error_kw={'elinewidth': 1.3, 'capthick': 1.3}
)
axes[1].axhline(0.5, color='0.7', ls='--', lw=1)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('AUC')
axes[1].set_xticks([0])
axes[1].set_xticklabels(['HIP+CTX'], rotation=20)
axes[1].text(
    0,
    min(ci_high + 0.04, 1.03),
    f'{auc:.3f}',
    ha='center',
    va='bottom',
    fontsize=9
)
axes[1].spines[['top', 'right']].set_visible(False)

fig.tight_layout()

out_pdf = f'{PLOT_DIR}/auc_summary_combined_green.pdf'
out_csv = f'{PLOT_DIR}/auc_summary_combined_green.csv'

fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.show()

pd.DataFrame({
    'task': ['HIP+CTX'],
    'auc': [auc],
    'ci_low': [ci_low],
    'ci_high': [ci_high],
    'n': [len(y)],
    'n_ad': [int(np.sum(y == 1))],
    'n_control': [int(np.sum(y == 0))],
}).to_csv(out_csv, index=False)

print(f'保存: {out_pdf}')
print(f'保存: {out_csv}')


In [ ]:
# ======================================================
# NVU-level AUC 超轻量版
# 直接使用 nvu_pred_prob，不再读取 nvu_latent.npy
# ======================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

PLOT_DIR = '../results/figure7'
PURPLE = '#7E57C2'

# 1. 读取 NVU 预测表
nvu_hip = pd.read_csv(f'{PLOT_DIR}/nvu_pred_hip.csv')
nvu_ctx = pd.read_csv(f'{PLOT_DIR}/nvu_pred_ctx.csv')

nvu_hip['tissue'] = 'hip'
nvu_ctx['tissue'] = 'ctx'

nvu = pd.concat([nvu_hip, nvu_ctx], ignore_index=True)

y = nvu['label'].astype(int).values
p = nvu['nvu_pred_prob'].astype(float).values
groups = nvu['sample_id'].astype(str).values

nvu_auc = roc_auc_score(y, p)
fpr, tpr, _ = roc_curve(y, p)

print(f'NVU-level AUC = {nvu_auc:.4f}')
print(f'NVU数量 = {len(nvu)}')
print(f'样本数 = {nvu["sample_id"].nunique()}')
print(f'AD NVU = {np.sum(y == 1)}')
print(f'Control NVU = {np.sum(y == 0)}')

# 2. sample-level cluster bootstrap CI
rng = np.random.default_rng(42)
unique_samples = np.unique(groups)
sample_to_idx = {
    s: np.where(groups == s)[0]
    for s in unique_samples
}

boot_aucs = []
N_BOOT = 1000

for _ in range(N_BOOT):
    sampled_samples = rng.choice(
        unique_samples,
        size=len(unique_samples),
        replace=True
    )
    idx = np.concatenate([sample_to_idx[s] for s in sampled_samples])

    if len(np.unique(y[idx])) < 2:
        continue

    boot_aucs.append(roc_auc_score(y[idx], p[idx]))

boot_aucs = np.array(boot_aucs)
ci_low, ci_high = np.percentile(boot_aucs, [2.5, 97.5])

print(f'95% CI = [{ci_low:.4f}, {ci_high:.4f}]')

# 3. 画图
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.5))

axes[0].plot(
    fpr, tpr,
    color=PURPLE,
    lw=2.5,
    label=f'NVU AUC={nvu_auc:.3f}'
)
axes[0].plot([0, 1], [0, 1], '--', color='0.7', lw=1)
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].legend(frameon=False, fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

yerr = np.array([[nvu_auc - ci_low], [ci_high - nvu_auc]])

axes[1].bar(
    [0],
    [nvu_auc],
    yerr=yerr,
    capsize=5,
    color=PURPLE,
    edgecolor='black',
    linewidth=0.7,
    error_kw={'elinewidth': 1.3, 'capthick': 1.3}
)
axes[1].axhline(0.5, color='0.7', ls='--', lw=1)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('AUC')
axes[1].set_xticks([0])
axes[1].set_xticklabels(['NVU-level'], rotation=20)
axes[1].text(
    0,
    min(ci_high + 0.04, 1.03),
    f'{nvu_auc:.3f}',
    ha='center',
    va='bottom',
    fontsize=9
)
axes[1].spines[['top', 'right']].set_visible(False)

fig.tight_layout()

out_pdf = f'{PLOT_DIR}/auc_nvu_level_direct_purple.pdf'
out_csv = f'{PLOT_DIR}/auc_nvu_level_direct_purple.csv'

fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.show()

pd.DataFrame({
    'task': ['NVU-level direct prediction'],
    'auc': [nvu_auc],
    'ci_low': [ci_low],
    'ci_high': [ci_high],
    'n_nvu': [len(nvu)],
    'n_samples': [len(unique_samples)],
    'n_ad_nvu': [int(np.sum(y == 1))],
    'n_control_nvu': [int(np.sum(y == 0))],
}).to_csv(out_csv, index=False)

print(f'保存: {out_pdf}')
print(f'保存: {out_csv}')


In [ ]:
# ======================================================
# 只画图：合并芯片metadata + 样本层latent图 + NVU层latent图
# 不重新训练模型
# ======================================================
import numpy as np
import pandas as pd

GNN_DIR = '../data/figure7'
PLOT_DIR = '../results/figure7'

# 如果你有自己整理好的metadata CSV，可以填这里；没有就用.py里内置的表
# CSV至少包含 sample_id 或 chip 列
METADATA_PATH = None
# METADATA_PATH = '../data/figure7/sample_metadata.csv'

# 1. 给已有预测表和latent meta表合并metadata并另存
metadata_outputs = save_prediction_tables_with_metadata(
    plot_dir=PLOT_DIR,
    metadata_path=METADATA_PATH,
)

print(metadata_outputs)

# 2. 样本分类展示：
#    用每个样本所有NVU latent的 mean + std 聚合成 sample-level representation
sample_latent_df = plot_sample_latent_embedding_from_saved(
    latent_paths={
        'hip': f'{GNN_DIR}/nvu_latent_hip.npy',
        'ctx': f'{GNN_DIR}/nvu_latent_ctx.npy',
    },
    meta_paths={
        'hip': f'{GNN_DIR}/nvu_latent_hip_meta.csv',
        'ctx': f'{GNN_DIR}/nvu_latent_ctx_meta.csv',
    },
    metadata_path=METADATA_PATH,
    out_prefix=f'{PLOT_DIR}/sample_latent_pca',
    method='pca',
    color_fields=[
        'group',
        'sex',
        'age_at_death',
        'braak_stage',
        'thal_stage',
        'clinical_severity',
        'sample_pred_prob',
        'nvu_pred_prob_mean',
    ],
)

display(sample_latent_df.head())



# 4. NVU级别 latent PCA：不下采样，每个点=一个NVU
nvu_latent_df = plot_nvu_latent_embedding_from_saved(
    latent_paths={
        'hip': f'{GNN_DIR}/nvu_latent_hip.npy',
        'ctx': f'{GNN_DIR}/nvu_latent_ctx.npy',
    },
    meta_paths={
        'hip': f'{GNN_DIR}/nvu_latent_hip_meta.csv',
        'ctx': f'{GNN_DIR}/nvu_latent_ctx_meta.csv',
    },
    metadata_path=METADATA_PATH,
    out_path=f'{PLOT_DIR}/nvu_latent_pca_all_points.pdf',
    method='pca',
)

display(nvu_latent_df.head())


In [ ]:
# ======================================================
# Figure7 sensitivity analysis：
# 1. 敏感基因
# 2. 敏感受配体对
# 3. 易感脑区 × NVU数量
# 海马 vs 皮层对比
# ======================================================
set_publication_plot_style()

gene_plot, lr_plot, region_plot = plot_sensitivity_analysis_panels(
    plot_dir=PLOT_DIR,
    out_prefix=f'{PLOT_DIR}/figure6_sensitivity',
    top_n_genes=20,
    top_n_lr=20,
    top_n_regions=12,
)

display(gene_plot.head())
display(lr_plot.head())
display(region_plot.head())


In [ ]:
import numpy as np
import pandas as pd

PLOT_DIR = '../results/figure7'

pred_hip = pd.read_csv(f'{PLOT_DIR}/gnn_pred_hip.csv')
pred_ctx = pd.read_csv(f'{PLOT_DIR}/gnn_pred_ctx.csv')
nvu_hip = pd.read_csv(f'{PLOT_DIR}/nvu_pred_hip.csv')
nvu_ctx = pd.read_csv(f'{PLOT_DIR}/nvu_pred_ctx.csv')

pred_hip['tissue'] = 'hip'
pred_ctx['tissue'] = 'ctx'
nvu_hip['tissue'] = 'hip'
nvu_ctx['tissue'] = 'ctx'

pred = pd.concat([pred_hip, pred_ctx], ignore_index=True)
nvu = pd.concat([nvu_hip, nvu_ctx], ignore_index=True)

pred['group'] = np.where(pred['label'].astype(int) == 1, 'AD', 'Control')
pred['pred_class'] = (pred['pred_prob'] >= 0.5).astype(int)
pred['correct'] = pred['pred_class'] == pred['label'].astype(int)
pred['abs_margin_from_0.5'] = (pred['pred_prob'] - 0.5).abs()

# NVU-level heterogeneity summary
nvu_stats = nvu.groupby(['sample_id', 'tissue']).agg(
    nvu_prob_mean=('nvu_pred_prob', 'mean'),
    nvu_prob_std=('nvu_pred_prob', 'std'),
    nvu_prob_min=('nvu_pred_prob', 'min'),
    nvu_prob_max=('nvu_pred_prob', 'max'),
    n_nvu=('nvu_pred_prob', 'size'),
    nvu_ad_like_frac=('nvu_pred_prob', lambda x: np.mean(x >= 0.7)),
    nvu_ctrl_like_frac=('nvu_pred_prob', lambda x: np.mean(x <= 0.3)),
    nvu_boundary_frac=('nvu_pred_prob', lambda x: np.mean((x > 0.3) & (x < 0.7))),
).reset_index().fillna(0)

df = pred.merge(nvu_stats, on=['sample_id', 'tissue'], how='left').fillna(0)

# 异质性分数：NVU预测越分裂越高；边界NVU越多也越高
df['heterogeneity_score'] = (
    df['nvu_prob_std'] +
    0.5 * df['nvu_boundary_frac'] +
    0.5 * np.minimum(df['nvu_ad_like_frac'], df['nvu_ctrl_like_frac'])
)

# 模型风险空间四象限
df['risk_quadrant'] = np.select(
    [
        (df['pred_prob'] >= 0.5) & (df['nvu_prob_mean'] >= 0.5),
        (df['pred_prob'] < 0.5) & (df['nvu_prob_mean'] < 0.5),
        (df['pred_prob'] >= 0.5) & (df['nvu_prob_mean'] < 0.5),
        (df['pred_prob'] < 0.5) & (df['nvu_prob_mean'] >= 0.5),
    ],
    [
        'sample_high_nvu_high',
        'sample_low_nvu_low',
        'sample_high_nvu_low',
        'sample_low_nvu_high',
    ],
    default='unknown'
)

# 1. 错分样本
misclassified = df[~df['correct']].copy()
misclassified = misclassified.sort_values(
    ['tissue', 'group', 'pred_prob'],
    ascending=[True, True, False]
)

# 2. 边界样本：最接近0.5
borderline = df.sort_values('abs_margin_from_0.5').copy()

# 3. 高异质性样本
heterogeneous = df.sort_values(
    'heterogeneity_score',
    ascending=False
).copy()

# 4. 高置信错分样本：最值得核查标签/质量
high_conf_wrong = misclassified.copy()
high_conf_wrong['wrong_confidence'] = np.where(
    high_conf_wrong['label'].astype(int) == 1,
    1 - high_conf_wrong['pred_prob'],
    high_conf_wrong['pred_prob']
)
high_conf_wrong = high_conf_wrong.sort_values(
    'wrong_confidence',
    ascending=False
)

cols = [
    'sample_id', 'tissue', 'group', 'label',
    'pred_prob', 'correct',
    'nvu_prob_mean', 'nvu_prob_std',
    'nvu_ad_like_frac', 'nvu_ctrl_like_frac',
    'nvu_boundary_frac',
    'heterogeneity_score',
    'risk_quadrant',
    'n_nvu',
]

print('\n================ 错分样本 ================')
display(misclassified[cols])

print('\n================ 高置信错分样本：优先核查 ================')
display(high_conf_wrong[cols + ['wrong_confidence']])

print('\n================ 最接近0.5的边界样本 ================')
display(borderline[cols].head(15))

print('\n================ NVU异质性最高样本 ================')
display(heterogeneous[cols].head(15))

# 保存
misclassified[cols].to_csv(f'{PLOT_DIR}/diagnostic_misclassified_samples.csv', index=False)
high_conf_wrong[cols + ['wrong_confidence']].to_csv(
    f'{PLOT_DIR}/diagnostic_high_conf_wrong_samples.csv',
    index=False
)
borderline[cols].to_csv(f'{PLOT_DIR}/diagnostic_borderline_samples.csv', index=False)
heterogeneous[cols].to_csv(f'{PLOT_DIR}/diagnostic_heterogeneous_samples.csv', index=False)

print('\n已保存:')
print(f'{PLOT_DIR}/diagnostic_misclassified_samples.csv')
print(f'{PLOT_DIR}/diagnostic_high_conf_wrong_samples.csv')
print(f'{PLOT_DIR}/diagnostic_borderline_samples.csv')
print(f'{PLOT_DIR}/diagnostic_heterogeneous_samples.csv')
